# English Lecture Audio to English/Korean STT & Summarization Pipeline
This notebook runs on Google Colab (with GPU) using OpenAI's Whisper model to transcribe and translate English lecture audio files dynamically into English and Korean STT texts, followed by English and Korean summarization.

In [ ]:
# 1. Install necessary libraries for audio transcription and summarization
!pip install -q openai-whisper ffmpeg-python transformers torch
import whisper
import os
import torch
from google.colab import files
from transformers import pipeline

print('Libraries loaded successfully!')

In [ ]:
# 2. Dynamic Audio File Upload (Runs interactively in Google Colab)
print('Please upload your lecture audio file (e.g., .m4a, .mp3, .wav):')
uploaded = files.upload()

# Automatically grab the uploaded filename and set dynamic output paths
input_audio_path = list(uploaded.keys())[0]
base_name = os.path.splitext(input_audio_path)[0]

output_stt_en_path = f"{base_name}-stt-en.txt"
output_stt_ko_path = f"{base_name}-stt-ko.txt"
output_summary_en_path = f"{base_name}-summary-en.txt"
output_summary_ko_path = f"{base_name}-summary-ko.txt"

print(f"\nTarget Input Audio: {input_audio_path}")

In [ ]:
# 3. Load Whisper Model and Generate English & Korean STT
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_size = "base"
print(f"Loading Whisper model ({model_size})...")
model = whisper.load_model(model_size, device=device)

print("Transcribing English audio (STT)...")
result_en = model.transcribe(input_audio_path, language="en", task="transcribe")
transcript_en = result_en["text"]

print("Translating & Transcribing into Korean STT...")
result_ko = model.transcribe(input_audio_path, language="en", task="translate")
transcript_ko = result_ko["text"]

print("STT generation completed!")

In [ ]:
# 4. Generate English and Korean Summaries using Transformers
print("Generating English Summary...")
summarizer_en = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if torch.cuda.is_available() else -1)

max_chunk_length = 1000
chunks_en = [transcript_en[i:i+max_chunk_length] for i in range(0, len(transcript_en), max_chunk_length)]

english_summaries = []
for chunk in chunks_en:
    if len(chunk.strip()) > 50:
        summary = summarizer_en(chunk, max_length=150, min_length=30, do_sample=False)
        english_summaries.append(summary[0]['summary_text'])

english_summary_text = "\n".join(english_summaries)

print("Generating Korean Summary...")
try:
    translator = pipeline("translation_en_to_ko", model="Helsinki-NLP/opus-mt-en-ko", device=0 if torch.cuda.is_available() else -1)
    korean_summary_chunks = []
    for chunk_text in english_summaries:
        translated = translator(chunk_text, max_length=200)
        korean_summary_chunks.append(translated[0]['translation_text'])
    korean_summary_text = "\n".join(korean_summary_chunks)
except Exception as e:
    korean_summary_text = f"Translation fallback error: {str(e)}\nEnglish Summary:\n{english_summary_text}"

print("Summarization completed!")

In [ ]:
# 5. Save and Download Results (STT + Summaries)
with open(output_stt_en_path, "w", encoding="utf-8") as f:
    f.write(transcript_en)

with open(output_stt_ko_path, "w", encoding="utf-8") as f:
    f.write(transcript_ko)

with open(output_summary_en_path, "w", encoding="utf-8") as f:
    f.write(english_summary_text)

with open(output_summary_ko_path, "w", encoding="utf-8") as f:
    f.write(korean_summary_text)

print(f"Successfully saved English STT: {output_stt_en_path}")
print(f"Successfully saved Korean STT: {output_stt_ko_path}")
print(f"Successfully saved English Summary: {output_summary_en_path}")
print(f"Successfully saved Korean Summary: {output_summary_ko_path}")

# Trigger browser downloads for all 4 files
files.download(output_stt_en_path)
files.download(output_stt_ko_path)
files.download(output_summary_en_path)
files.download(output_summary_ko_path)

# Display previews
print("\n--- English Summary Preview ---")
print(english_summary_text[:300])
print("\n--- Korean Summary Preview ---")
print(korean_summary_text[:300])